# Asian options: control variate variance reduction

Checks on `pricing.exotics`:

1. **A single simulated path**: what the arithmetic and geometric running averages actually look like, to build intuition before anything else.
2. **Control variate benefit**: standard error with vs. without the geometric control variate, at a fixed number of paths.
3. **Convergence**: standard error vs. N (log-log), plain vs. control-variate, both should scale as 1/sqrt(N) but the control-variate curve should sit far below.
4. **Sanity ordering**: geometric <= arithmetic <= vanilla price (call), and the reverse geometric/arithmetic inequality for a put.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pricing.black_scholes import black_scholes_price
from pricing.exotics import asian_arithmetic_mc_price, geometric_asian_price

In [ ]:
# Market parameters (same as previous notebooks, for comparability)
S0 = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.20
q = 0.02
n_fixings = 50  # e.g. roughly weekly averaging over a 1-year option

## 1. What the running averages look like on a single path

In [ ]:
rng_path = np.random.default_rng(7)
dt = T / n_fixings
z = rng_path.standard_normal(n_fixings)
log_increments = (r - q - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * z
path = S0 * np.exp(np.cumsum(log_increments))

running_arithmetic = np.cumsum(path) / np.arange(1, n_fixings + 1)
running_geometric = np.exp(np.cumsum(np.log(path)) / np.arange(1, n_fixings + 1))
fixing_dates = np.linspace(T / n_fixings, T, n_fixings)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(fixing_dates, path, label="Underlying $S_t$ at each fixing", alpha=0.6)
ax.plot(fixing_dates, running_arithmetic, label="Running arithmetic average", linewidth=2)
ax.plot(fixing_dates, running_geometric, "--", label="Running geometric average", linewidth=2)
ax.axhline(K, color="gray", linestyle="--", linewidth=0.8, label="Strike K")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Price")
ax.set_title("One simulated path: arithmetic vs. geometric running average")
ax.legend()
fig.tight_layout()
plt.show()

print(f"Final arithmetic average: {running_arithmetic[-1]:.4f}")
print(f"Final geometric average:  {running_geometric[-1]:.4f}  (always <= arithmetic, AM-GM)")

## 2. Control variate benefit at a fixed N

In [ ]:
plain = asian_arithmetic_mc_price(
    S0, K, T, r, sigma, n_fixings, option_type="call", q=q,
    n_paths=100_000, use_control_variate=False, seed=42,
)
control_variate = asian_arithmetic_mc_price(
    S0, K, T, r, sigma, n_fixings, option_type="call", q=q,
    n_paths=100_000, use_control_variate=True, seed=42,
)

print(f"Plain MC:           price={plain.price:.6f}, SE={plain.std_error:.6f}")
print(f"Control variate MC: price={control_variate.price:.6f}, SE={control_variate.std_error:.6f}")
print(f"Variance reduction factor (SE_plain / SE_cv): {plain.std_error / control_variate.std_error:.2f}x")

## 3. Convergence: standard error vs. N

In [ ]:
n_paths_values = np.unique(np.logspace(2, 6, 13).astype(int))

rows = []
for n_paths in n_paths_values:
    res_plain = asian_arithmetic_mc_price(
        S0, K, T, r, sigma, n_fixings, option_type="call", q=q,
        n_paths=n_paths, use_control_variate=False, seed=42,
    )
    res_cv = asian_arithmetic_mc_price(
        S0, K, T, r, sigma, n_fixings, option_type="call", q=q,
        n_paths=n_paths, use_control_variate=True, seed=42,
    )
    rows.append({
        "n_paths": n_paths,
        "se_plain": res_plain.std_error,
        "se_control_variate": res_cv.std_error,
        "variance_reduction_factor": res_plain.std_error / res_cv.std_error,
    })

df = pd.DataFrame(rows)
df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(df["n_paths"], df["se_plain"], "o-", label="SE (plain)")
ax.loglog(df["n_paths"], df["se_control_variate"], "o-", label="SE (control variate)")

C = df["se_plain"].iloc[0] * np.sqrt(df["n_paths"].iloc[0])
ax.loglog(df["n_paths"], C / np.sqrt(df["n_paths"]), "k--", label="Slope -1/2 reference")

ax.set_xlabel("Number of paths N")
ax.set_ylabel("Standard error")
ax.set_title("Asian option Monte Carlo: SE vs N, plain vs. control variate")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Sanity ordering: geometric vs. arithmetic vs. vanilla

In [ ]:
for option_type in ["call", "put"]:
    vanilla_price = black_scholes_price(S0, K, T, r, sigma, option_type=option_type, q=q)
    geometric_price = geometric_asian_price(S0, K, T, r, sigma, n_fixings, option_type=option_type, q=q)
    arithmetic_price = asian_arithmetic_mc_price(
        S0, K, T, r, sigma, n_fixings, option_type=option_type, q=q, n_paths=200_000, seed=42
    ).price
    print(f"{option_type:5s}  vanilla={vanilla_price:.6f}  geometric={geometric_price:.6f}  arithmetic={arithmetic_price:.6f}")

print()
print("Expected: for a call, geometric < arithmetic < vanilla (averaging reduces effective volatility,")
print("and the geometric average is never above the arithmetic one, pathwise, by AM-GM).")
print("For a put, the AM-GM inequality flips direction: geometric > arithmetic (still < vanilla).")